Build a RAG system over a real document set (Confluence export, GitHub docs, or Wikipedia subset)
• Implement 2 chunking strategies and benchmark them against the same 30-question test set
• Add metadata filters (by date range, document type)
Implement hybrid search and measure precision@5 vs pure dense retrieval

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader
)

documents = []

data_dir = Path("./data")

for file in data_dir.iterdir():

    if file.suffix.lower() == ".pdf":
        loader = PyPDFLoader(str(file))
        docs = loader.load()

        for doc in docs:
            doc.metadata["doc_type"] = "research_paper"
            doc.metadata["file_name"] = file.name
            doc.metadata["date"]= "2025-09-18"

        documents.extend(docs)

    elif file.suffix.lower() == ".txt":
        loader = TextLoader(str(file))
        docs = loader.load()

        for doc in docs:
            doc.metadata["doc_type"] = "txt_document"
            doc.metadata["file_name"] = file.name
            doc.metadata["date"]= "2025-09-18"

        documents.extend(docs)

print(f"Loaded {len(documents)} document chunks/pages")

/tmp/ipykernel_2152/3405484655.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 12 document chunks/pages


In [3]:
print(documents[0].metadata)

{'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'PyPDF', 'creationdate': '2025-09-18T03:03:54+02:00', 'moddate': '2025-09-18T03:03:54+02:00', 'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'doc_type': 'research_paper', 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf', 'date': '2025-09-18'}


In [ ]:
import re

def clean_text(text):
    
    text = re.sub(r"/C\d+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [3]:
for doc in documents:
    doc.page_content = clean_text(doc.page_content)
print(documents[0].page_content[:500])
print(documents[0].metadata)

CATCHWORD Retrieval-Augmented Generation (RAG) Michael Klesel • H. Felix Wittmann Received: 22 July 2024 / Accepted: 7 April 2025 / Published online: 1 June 2025 The Author(s) 2025 Keywords Retrieval-augmented generation Artiﬁcial intelligence Large language models Information retrieval 1 Introduction The necessity for information is a fundamental aspect of human nature, and as such, there are ongoing efforts to enhance information retrieval with information systems (Alavi and Leidner 2001; Alav
{'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'PyPDF', 'creationdate': '2025-09-18T03:03:54+02:00', 'moddate': '2025-09-18T03:03:54+02:00', 'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'doc_type': 'research_paper', 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf', 'date': '2025-09-18'}


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter( chunk_size=500, chunk_overlap=0)

recursive_chunks= recursive_splitter.split_documents(documents)

seen = set()
filtered_chunks = []

for doc in recursive_chunks:

    text = doc.page_content.strip()

    if len(text) < 250:
        continue

    if text in seen:
        continue

    seen.add(text)
    filtered_chunks.append(doc)

print("Original chunks:", len(recursive_chunks))
print("Filtered chunks:", len(filtered_chunks))
# print(f"recursiveChunks:{len(recursive_chunks)}")


Original chunks: 128
Filtered chunks: 122


# Chunking in Semantic 

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name ="sentence-transformers/all-MiniLM-L6-v2")
semantic_splitter= SemanticChunker(embed_model)

semantic_chunks = semantic_splitter.split_documents(documents)

seen = set()
filtered_chunk = []

for doc in semantic_chunks:

    text = doc.page_content.strip()

    if len(text) < 250:
        continue

    if text in seen:
        continue

    seen.add(text)
    filtered_chunk.append(doc)

print("Original chunks:", len(semantic_chunks))
print("Filtered chunks:", len(filtered_chunk))
# print(f"semantic Chunks: {len(semantic_chunks)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 427.25it/s]


Original chunks: 45
Filtered chunks: 26


# recursive vector storage

In [33]:
from langchain_chroma import Chroma

recursive_db= Chroma.from_documents(
    documents=filtered_chunks,
    embedding=embed_model,
    persist_directory="./recursive_dbv1"
)


# semantic db

In [47]:

semantic_db = Chroma.from_documents(
    documents=filtered_chunk,
    embedding=embed_model,
    persist_directory="./semantic_dbv2"
)

# recursive retriever

In [34]:
recursive_retriever = recursive_db.as_retriever(search_kwargs={"k":5})

In [ ]:
query = "What are the benefits of RAG?"

results = recursive_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content[:1000])



Result 1
answered using this data, reducing the likelihood of inaccurate answers. Thus, RAG is an effective measure to enhance factual accuracy. A RAG architecture allows references to be provided to the contextual data stored in the vector database. Providing valid references to the generated result has been termed grounding (Magesh et al. 2024). Grounding is a signiﬁcant advantage, because it gives a user additional information about where the information comes from. Therefore, a user looking for

Result 2
needed that investigates what outcomes can be expected from the evolution and advancements of new architectures (Haki et al. 2020). Ultimately, organizations seek opportunities to increase efﬁciency and productivity. RAG has the potential to improve business processes and enhance organizational decision making. Nevertheless, it remains unclear to what extent organizations can beneﬁt from using RAG. In addition, RAG can also help organizations meet regulatory requirements. For exam

In [36]:
results = recursive_db.similarity_search_with_score(
    query,
    k=5
)

for i, (doc, score) in enumerate(results):
    print(f"\nResult {i+1}")
    print("Score:", score)
    print(doc.page_content[:300])


Result 1
Score: 0.8369678854942322
answered using this data, reducing the likelihood of inaccurate answers. Thus, RAG is an effective measure to enhance factual accuracy. A RAG architecture allows references to be provided to the contextual data stored in the vector database. Providing valid references to the generated result has bee

Result 2
Score: 0.9287424087524414
needed that investigates what outcomes can be expected from the evolution and advancements of new architectures (Haki et al. 2020). Ultimately, organizations seek opportunities to increase efﬁciency and productivity. RAG has the potential to improve business processes and enhance organizational deci

Result 3
Score: 0.9835426807403564
(Bruch et al. 2023) (Table 3). 4 Implications for BISE Researchers This catchword article seeks to provide a fundamental overview of RAG, highlight characteristics of RAG architectures, and outline implications of RAG. Since prior work has already identiﬁed important avenues for research w

# metadata filter

In [50]:
results = recursive_db.similarity_search(
    query="What are the benefits of RAG?",
    k=5,
    filter={
        "document_type": "research_paper"
    }
)

In [55]:
# import date
retrieved_docs = recursive_retriever.invoke(query)
# print(retrieved_docs[0].metadata)
filtered_docs = []

for doc in retrieved_docs:

     doc_date = doc.metadata["date"]

     if "2025-01-01" <= doc_date <= "2025-12-31":
         filtered_docs.append(doc)
print(len(filtered_docs))

5


In [85]:
import time

start = time.time()

recursive_retriever.invoke(query)

end = time.time()

recursive_time = end - start
print(f"recursive time: {recursive_time}")

recursive time: 0.06633996963500977


In [44]:
import boto3
import json
import os
from dotenv import load_dotenv

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv( "AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

load_dotenv("myenv.env")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [45]:
context = "\n\n".join([doc.page_content for doc in  filtered_docs])

prompt =f"""Answer using onlt the context below

context:{context}

Question:{query}
"""
body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.2
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)
response = json.loads(response["body"].read())


print(response["output"]["message"]["content"][0]["text"])



The benefits of Retrieval-Augmented Generation (RAG) as outlined in the provided context include:

1. **Enhanced Factual Accuracy**: RAG is effective in improving the factual accuracy of generated results by allowing references to be provided to the contextual data stored in the vector database. This grounding process ensures that the information is traceable and verifiable.

2. **Improved Decision Making**: RAG has the potential to enhance organizational decision-making processes by providing accurate and contextually relevant information.

3. **Increased Efficiency and Productivity**: Organizations can leverage RAG to improve business processes, thereby increasing overall efficiency and productivity.

4. **Regulatory Compliance**: RAG can help organizations meet regulatory requirements by providing traceable and verifiable information.

5. **Support for Research**: RAG offers nuanced research questions and avenues for Business Information Systems Engineering (BISE) researchers, contr

# semantic Retriever

In [86]:
semantic_retriever = semantic_db.as_retriever(search_kwargs={"k":8})

In [87]:
query = "What are the benefits of RAG?"

results = semantic_retriever.invoke(query)

for doc in results:
    print(doc.page_content[:200])
    print(doc.metadata)
    print("-" * 50)

( 2022) or Gallegos et al. (2024). Blinkered chunk effect (BCE) A plain vanilla RAG implementation is limited in terms of a comprehensive understanding of extensive data. New approaches such as RAPTOR
{'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'date': '2025-09-18', 'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'creationdate': '2025-09-18T03:03:54+02:00', 'creator': 'PyPDF', 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf', 'page_label': '8', 'moddate': '2025-09-18T03:03:54+02:00', 'total_pages': 12, 'doc_type': 'research_paper', 'page': 7}
--------------------------------------------------
When LLMs try to answer questions for a speciﬁc domain that is not part of the training data, hallucinations are likely. One way to address this problem is to ﬁne-tune the LLM, which is less expensive
{'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf', 'moddate': '2025-09-18T03

In [88]:
doc.metadata

{'creator': 'PyPDF',
 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf',
 'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)',
 'total_pages': 12,
 'moddate': '2025-09-18T03:03:54+02:00',
 'date': '2025-09-18',
 'page_label': '5',
 'source': 'data/Retrieval-Augmented_Generation_RAG.pdf',
 'creationdate': '2025-09-18T03:03:54+02:00',
 'doc_type': 'research_paper',
 'page': 4}

In [89]:
results = semantic_db.similarity_search(
    query,
    k=5,
    filter={
        "document_type": "research_paper"
    }
)

In [90]:
retrieved_docs = semantic_retriever.invoke(query)

filtered_docs = []

for doc in retrieved_docs:

    doc_date = doc.metadata.get("date")

    if doc_date and "2025-01-01" <= doc_date <= "2025-12-31":
        filtered_docs.append(doc)

print(len(filtered_docs))

8


In [91]:
import time

start = time.time()

semantic_retriever.invoke(query)

semantic_time = time.time() - start

print(f"Semantic Retrieval Time: {semantic_time:.4f}")

Semantic Retrieval Time: 0.1555


retrieval time is = How long it takes to find relevant chunks

In [92]:
test_queries = [
    "What is RAG?",
    "How does retrieval improve LLMs?",
    "What are vector databases?",
    "What is semantic search?",
    "What are embeddings?",
    
    "What is a large language model?",
    "How do large language models generate text?",
    "What are the limitations of large language models?",
    "Why do LLMs hallucinate?",
    "How can hallucinations be reduced in LLMs?",
    
    "What are embeddings?",
    "How are text embeddings created?",
    "Why are embeddings important for semantic search?",
    "What is the difference between embeddings and keywords?",
    "How are embeddings used in vector databases?",
    
    "What is a vector database?",
    "Why are vector databases used in RAG systems?",
    "How does similarity search work in a vector database?",
    "What is cosine similarity?",
    "How are vectors stored and retrieved?",
    
    "What is semantic search?",
    "How does semantic search differ from keyword search?",
    "What are the advantages of semantic search?",
    "How does semantic chunking work?",
    "Why can semantic chunking improve retrieval quality?",
    
    "What is document chunking?",
    "Why is chunk overlap used in text splitting?",
    "What is recursive chunking?",
    "How do recursive and semantic chunking differ?",
    "What factors affect retrieval performance in RAG systems?"
    
]

In [93]:
import time

recursive_times = []
semantic_times = []

for query in test_queries:

    start = time.time()
    recursive_retriever.invoke(query)
    recursive_times.append(time.time() - start)

    start = time.time()
    semantic_retriever.invoke(query)
    semantic_times.append(time.time() - start)

print("Average Recursive:",
      sum(recursive_times)/len(recursive_times))

print("Average Semantic:",
      sum(semantic_times)/len(semantic_times))

Average Recursive: 0.04527886708577474
Average Semantic: 0.04035046100616455


# Hybrid search

In [59]:
def dense_search(query, top_k=10):
    results = semantic_db.similarity_search(
        query,
        k=top_k
    )
    return results

In [60]:
results = dense_search(
    "What are benifits of RAG?",
    top_k=10
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print("Length:", len(doc.page_content))
    print(doc.page_content[:500])


Result 1
Length: 2428
When LLMs try to answer questions for a speciﬁc domain that is not part of the training data, hallucinations are likely. One way to address this problem is to ﬁne-tune the LLM, which is less expensive than building a founda- tion model but requires considerable resources nonetheless. Studies have shown that ﬁne-tuning can also lead to hal- lucinations (Gekhman et al. 2024). On the other hand, RAG is an effective way to add this knowledge. By adding additional information, questions can be answer

Result 2
Length: 595
( 2022) or Gallegos et al. (2024). Blinkered chunk effect (BCE) A plain vanilla RAG implementation is limited in terms of a comprehensive understanding of extensive data. New approaches such as RAPTOR (Sarthi et al. 2024) are required to reduce this issue. Retrieval effectiveness The effectiveness of a RAG architecture depends on how effectively the retrieval mechanism works. This includes the effectiveness of the document ranking (i.e., are the most

In [61]:
docs = semantic_db.get()["documents"]

print("Total docs:", len(docs))

for i in range(10):
    print(f"\nChunk {i}")
    print("Length:", len(docs[i]))
    print(docs[i][:300])

Total docs: 26

Chunk 0
Length: 3150
CATCHWORD Retrieval-Augmented Generation (RAG) Michael Klesel • H. Felix Wittmann Received: 22 July 2024 / Accepted: 7 April 2025 / Published online: 1 June 2025 The Author(s) 2025 Keywords Retrieval-augmented generation Artiﬁcial intelligence Large language models Information retrieval 1 Introducti

Chunk 1
Length: 680
Wittmann Frankfurt University of Applied Sciences, Nibelungenplatz 1, Frankfurt, Germany e-mail: michael.klesel@fra-uas.de M. Klesel Hessian Center for AI (hessian.AI), Darmstadt, Germany 1 Recently, the literature has suggested bullshit as a more appropriate term, since there is no concept of truth

Chunk 2
Length: 1297
from within an organization and reduces the risk of hal- lucinations. This new architecture offers important advancements compared to previous architectures and presents new challenges for research and academia. Previous catchword articles have already covered important aspects of AI, namely fair AI

Chunk 3
Length: 

# BM25 with semantic 

In [62]:
texts = [
    doc.page_content
    for doc in filtered_chunk
]
from rank_bm25 import BM25Okapi

token_corpus = [
    text.lower().split()
    for text in texts
]

bm25 = BM25Okapi(token_corpus)

In [63]:
def bm25_search(query, top_k=10):

    token_query = query.lower().split()

    scores = bm25.get_scores(token_query)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:top_k]:

        results.append({
            "document": texts[idx],
            "score": score
        })

    return results

In [72]:
results = bm25_search(
    "What are benifits of RAG?",
    top_k=10
)

for i, result in enumerate(results):
    print(f"\nResult {i+1}")
    print("Score:", result["score"])
    print(result["document"][:500])


Result 1
Score: 3.8198420077770825
Wikipedia and may be more effective in ﬁnding relevant information. Example research questions that emerge with RAG-based systems relevant for individual-level BISE research are Can grounding-based explanations outper- form traditional XAI approaches in improving user trust? or To what extent does RAG enhance individuals’ work- place performance? Finally, RAG also invites more research that investi- gates the economic value of new information systems architectures. More generally, research is ne

Result 2
Score: 3.6967597191810895
tuning that leads to superior organizational performance? , or To what extent does a RAG-based architecture con- tribute to better IT business alignment? Secondly, individuals interacting with RAG-based sys- tems (e.g., using a CA) will experience changes in how results are presented. Most importantly, RAG offers the ability to add references to contextual data, which has been coined ‘ ‘grounding’ ’ (Magesh et al. 2024). Pr

# Hybrid Search

In [67]:
def hybrid_search(query, top_k=10):

    dense_results = dense_search(query, top_k)

    bm25_results = bm25_search(query, top_k)

    combined_docs = []

    for doc in dense_results:
        combined_docs.append(doc.page_content)

    for item in bm25_results:
        combined_docs.append(item["document"])

    unique_docs = list(dict.fromkeys(combined_docs))

    return unique_docs[:top_k]

In [77]:
query ="What are benefits of RAG?"
docs = hybrid_search(query)

print("Retrieved Documents:", len(docs))

for i, doc in enumerate(docs[:10]):
    print(f"\n===== Retrieved Chunk {i+1} =====")
    print(doc[:300])

    

Retrieved Documents: 10

===== Retrieved Chunk 1 =====
When LLMs try to answer questions for a speciﬁc domain that is not part of the training data, hallucinations are likely. One way to address this problem is to ﬁne-tune the LLM, which is less expensive than building a founda- tion model but requires considerable resources nonetheless. Studies have sh

===== Retrieved Chunk 2 =====
( 2022) or Gallegos et al. (2024). Blinkered chunk effect (BCE) A plain vanilla RAG implementation is limited in terms of a comprehensive understanding of extensive data. New approaches such as RAPTOR (Sarthi et al. 2024) are required to reduce this issue. Retrieval effectiveness The effectiveness o

===== Retrieved Chunk 3 =====
2007). The following research questions are examples of BISE scholars conducting research at the organizational level: Do high levels of data management capability, e.g., Data Mesh including RAG, lead to high levels of organizational performance? , What is the optimal balance of d

In [79]:
import json

query = "What are benefits of RAG?"
docs = hybrid_search(query)

labels = []

for i, chunk in enumerate(docs[:10]):

    prompt = f"""
You are evaluating retrieval quality.

User Query:
{query}

Retrieved Chunk:
{chunk}

Task:
Does this chunk contain information that would help answer the query?

Return exactly one character:

1 = Yes
0 = No
"""

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 5,
            "temperature": 0.0
        }
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    response = json.loads(response["body"].read())

    label = response["output"]["message"]["content"][0]["text"].strip()

    try:
        label = int(label)
    except:
        label = 0

    labels.append(label)

    print(f"Chunk {i+1}: {label}")

print("\nLabels:", labels)

precision_at_5 = sum(labels[:5]) / 5
precision_at_10 = sum(labels) / 10

print("Precision@5 :", precision_at_5)
print("Precision@10:", precision_at_10)

Chunk 1: 1
Chunk 2: 0
Chunk 3: 1
Chunk 4: 1
Chunk 5: 0
Chunk 6: 1
Chunk 7: 1
Chunk 8: 1
Chunk 9: 1
Chunk 10: 1

Labels: [1, 0, 1, 1, 0, 1, 1, 1, 1, 1]
Precision@5 : 0.6
Precision@10: 0.8


# pure dense retrieval


In [80]:
def dense_search(query, top_k=10):
    results = semantic_db.similarity_search(
        query,
        k=top_k
    )
    return results

results = dense_search(
    "What are benefits of RAG",
    top_k=10
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print("Length:", len(doc.page_content))
    print(doc.page_content[:500])



Result 1
Length: 595
( 2022) or Gallegos et al. (2024). Blinkered chunk effect (BCE) A plain vanilla RAG implementation is limited in terms of a comprehensive understanding of extensive data. New approaches such as RAPTOR (Sarthi et al. 2024) are required to reduce this issue. Retrieval effectiveness The effectiveness of a RAG architecture depends on how effectively the retrieval mechanism works. This includes the effectiveness of the document ranking (i.e., are the most relevant documents ranked ﬁrst?) and how well

Result 2
Length: 2428
When LLMs try to answer questions for a speciﬁc domain that is not part of the training data, hallucinations are likely. One way to address this problem is to ﬁne-tune the LLM, which is less expensive than building a founda- tion model but requires considerable resources nonetheless. Studies have shown that ﬁne-tuning can also lead to hal- lucinations (Gekhman et al. 2024). On the other hand, RAG is an effective way to add this knowledge. By adding a

In [81]:
import json

query = "What are benefits of RAG?"
docs = dense_search(query)

labels = []

for i, chunk in enumerate(docs[:10]):

    prompt = f"""
You are evaluating retrieval quality.

User Query:
{query}

Retrieved Chunk:
{chunk}

Task:
Does this chunk contain information that would help answer the query?

Return exactly one character:

1 = Yes
0 = No
"""

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 5,
            "temperature": 0.0
        }
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    response = json.loads(response["body"].read())

    label = response["output"]["message"]["content"][0]["text"].strip()

    try:
        label = int(label)
    except:
        label = 0

    labels.append(label)

    print(f"Chunk {i+1}: {label}")

print("\nLabels:", labels)

precision_at_5 = sum(labels[:5]) / 5
precision_at_10 = sum(labels) / 10

print("Precision@5 :", precision_at_5)
print("Precision@10:", precision_at_10)

Chunk 1: 1
Chunk 2: 1
Chunk 3: 1
Chunk 4: 1
Chunk 5: 1
Chunk 6: 1
Chunk 7: 1
Chunk 8: 1
Chunk 9: 1
Chunk 10: 1

Labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Precision@5 : 1.0
Precision@10: 1.0
